# 01 — Prepare data: inputs, outputs, and their shapes

The goal of this notebook is **understanding**, not training. We build the
model's **inputs** (`X`) and **outputs** (`y`) one piece at a time, printing
the shape after every step and drawing a picture wherever it helps, so the
preprocessing is fully legible. The last section *materializes* the prepared
samples to a fast on-disk cache that notebook **02** trains on.

**The learning problem (v1).** For each calendar **day D** with at least one
reported flood (**any** of the three layers — see §2), predict *which **50 km
cells** of CONUS flood that day* from **the previous day's (D−1) GOES satellite
imagery + lightning**.

| | what | shape | meaning |
|---|---|---|---|
| **input** GOES | prev-day band sequence | `(T=6, 6, 1500, 2500)` | 6 daytime frames × 6 ABI bands on the 2 km grid |
| **input** GLM | prev-day lightning | `(1500, 2500)` | one **whole-day** flash-density map (all 24 h) |
| **output** `y` | flood map for day D | `(59, 95)` | 0/1 per **50 km** cell over CONUS land |

Everything below makes those tensors concrete.

## 0. Config & imports

Paths, the bands we keep, and the time window. Nothing is computed here — just the knobs in one place.

In [ ]:
import json
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import netCDF4
import numpy as np
import pandas as pd

# all shared constants live in the repo-root config.py (single source of truth)
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from config import (BANDS, CACHE_DIR, CELL_KM, DATA_DIR, GLM_DIR, IMG_H, IMG_W,
                    N_BAND, SPLIT_FRACS, SPLIT_SEED, STATS_PATH, T_FRAMES,
                    UNIFIED_PARQUET, YEAR, build_grid_cells)

# inputs: a 6-frame GOES band sequence + ONE whole-day lightning map
print(f"GOES seq : (T={T_FRAMES}, {N_BAND}, {IMG_H}, {IMG_W})  "
      f"= {T_FRAMES*N_BAND*IMG_H*IMG_W*4/1e6:.0f} MB as float32")
print(f"GLM map  : ({IMG_H}, {IMG_W})  whole-day flash density")
print(f"output   : {CELL_KM} km cells | bands {BANDS} | {YEAR}")

## 1. The output target — 50 km cells

The output grid is **generated fresh** by `config.build_grid_cells()`: square
cells **`CELL_KM` km on a side** (=50) are laid over CONUS in an equal-area
projection (EPSG:5070, so they're genuinely 50 km) and kept where they
intersect US land. **Nothing is stored** — it's a pure function of `CELL_KM` +
the CONUS boundary, so notebook 02 rebuilds the identical grid. `cells` are
those polygons tagged with `(R, C)`; notebook 02 maps GOES pixels onto them.
The output `y` is a `(59, 95)` 0/1 image.

> **Orientation:** row 0 = north, col 0 = west — matching the GOES imagery, so
> inputs and labels share one orientation and plots render right-way-up.

In [ ]:
# generated fresh at CELL_KM over CONUS land (config.build_grid_cells; nothing stored)
cells, GRID_R, GRID_C, land_mask = build_grid_cells()

print(f"{CELL_KM} km grid: ({GRID_R}, {GRID_C})  <- every y map is this shape")
print(f"land cells: {int(land_mask.sum())} of {GRID_R*GRID_C}  <- loss computed here")

plt.figure(figsize=(6, 4))
plt.imshow(land_mask, cmap="Greys", interpolation="none")
plt.title(f"land mask  ({GRID_R}x{GRID_C}, {int(land_mask.sum())} cells)")
plt.xlabel("col (W->E)"); plt.ylabel("row (N->S)"); plt.tight_layout()

## 2. Labels — all three flood layers → daily 0/1 maps

We use **every** layer of the unified flood frame as a positive — for a bigger,
denser target:

- **groundsource** — flood extents from local news reports (Gemini-extracted)
- **NWS warnings** — forecaster-issued Flash/Areal Flood *warning* polygons
- **storm events** — NCEI human-*confirmed* flood occurrences

We take each event's footprint, find which 50 km cells it **intersects**, and
stamp a 1 on the event's start day (`issue_date`). The result is a dict
`labels[day] -> (59, 95)`.

> The three layers differ in nature (observed vs predicted vs confirmed), so
> unioning them broadens coverage but mixes label semantics — a deliberate v1
> choice to maximize positives.

In [ ]:
u = gpd.read_parquet(UNIFIED_PARQUET)
ev = u[u["issue_date"].dt.year == 2021].copy()         # ALL sources
ev["label_day"] = ev["issue_date"].dt.normalize()      # day resolution

# spatial join: which 50 km cell does each event footprint touch?
j = gpd.sjoin(cells, ev[["label_day", "geometry"]], predicate="intersects")

labels = {}
for day, g in j.groupby("label_day"):
    a = np.zeros((GRID_R, GRID_C), dtype=np.float32)
    a[g["R"], g["C"]] = 1.0
    labels[day.date()] = a

pos_per_day = np.array([a[land_mask].sum() for a in labels.values()])
print(f"{len(ev):,} flood events (all layers) on {len(labels)} days in {YEAR}")
print("by source:", ev["source"].value_counts().to_dict())
print(f"flooded land cells/day: median {np.median(pos_per_day):.0f}, "
      f"max {pos_per_day.max():.0f} "
      f"({np.median(pos_per_day)/land_mask.sum():.1%} of land)")

In [ ]:
# look at the busiest flood day's label map
busy_day = max(labels, key=lambda d: labels[d].sum())
ymap = labels[busy_day]
shown = np.where(land_mask, ymap, np.nan)          # grey-out ocean for clarity

plt.figure(figsize=(6, 4))
plt.imshow(np.where(land_mask, 0.15, np.nan), cmap="Greys", vmin=0, vmax=1)
plt.imshow(shown, cmap="Reds", vmin=0, vmax=1, interpolation="none")
plt.title(f"y for {busy_day}  ->  shape {ymap.shape}, "
          f"{int(ymap.sum())} flooded cells")
plt.xlabel("col"); plt.ylabel("row"); plt.tight_layout()

## 3. Input part A — GOES ABI bands

GOES files are one NetCDF per scan (`MCMIPC`, all 16 bands inside). We keep
**6 frames per day** (the 16–21 UTC daytime window) and **6 bands** per frame.
First, the helpers that find a day's files and read their scan time from the
filename.

In [ ]:
def _scan_token(p):
    "Return the s-token (s{YYYYDDDHHMM...}) from a GOES filename."
    for part in p.name.split("_"):
        if part.startswith("s") and part[1:].isdigit():
            return part
    return p.name


def _scan_dt(p):
    "Scan-start datetime parsed from the filename token (UTC)."
    return datetime.strptime(_scan_token(p)[1:12], "%Y%j%H%M")


def goes_files(d):
    "Sorted list of the day's GOES NetCDFs (GOES16 or GOES19 auto-globbed)."
    pat = f"*/{d.year}/{d.month:02d}/{d.day:02d}/*.nc"
    return sorted(DATA_DIR.glob(pat), key=_scan_token)


def glm_path(d):
    return GLM_DIR / str(d.year) / f"glm_flashes_{d:%Y%m%d}.parquet"


# pick one example day that has a flood the NEXT day (so it's a real sample)
example_label_day = sorted(d for d in labels
                           if len(goes_files(d - timedelta(days=1))) == T_FRAMES
                           and glm_path(d - timedelta(days=1)).exists())[5]
example_in_day = example_label_day - timedelta(days=1)
files = goes_files(example_in_day)
print(f"input day {example_in_day}  ->  label day {example_label_day}")
print(f"{len(files)} frames; scan times (UTC):",
      [f"{_scan_dt(f):%H:%M}" for f in files])

In [ ]:
# read ONE band from ONE frame to see the raw values
mid = files[3]                                   # a midday frame
with netCDF4.Dataset(mid) as nc:
    raw13 = np.ma.filled(nc["CMI_C13"][:], np.nan)    # clean-IR brightness temp (K)
print(f"one band, one frame -> shape {raw13.shape}, dtype {raw13.dtype}")
print(f"band 13 (cloud-top temp): min {np.nanmin(raw13):.0f} K, "
      f"max {np.nanmax(raw13):.0f} K  (cold tops = tall storms)")

plt.figure(figsize=(8, 4.6))
plt.imshow(raw13[::4, ::4], cmap="gray_r")       # subsample just for display
plt.title(f"GOES band 13 (clean IR)  {example_in_day} {_scan_dt(mid):%H:%M}Z")
plt.colorbar(label="brightness temp (K)", shrink=0.8)
plt.xticks([]); plt.yticks([]); plt.tight_layout()

### Normalization

Raw band values live on wildly different scales (visible reflectance 0–1,
brightness temperatures ~180–320 K). We standardize each band to ~zero mean /
unit variance using **per-band statistics** computed from a handful of training
frames (cached to JSON so re-runs are instant).

In [ ]:
if STATS_PATH.exists():
    stats = json.loads(STATS_PATH.read_text())
    print(f"loaded cached band stats from {STATS_PATH.name}")
else:
    # sample a few midday frames across the year and measure mean/std per band
    cand = sorted(d - timedelta(days=1) for d in labels
                  if len(goes_files(d - timedelta(days=1))) == T_FRAMES)
    rng = np.random.default_rng(0)
    pick = rng.choice(len(cand), size=8, replace=False)
    acc = {b: [] for b in BANDS}
    for i in pick:
        with netCDF4.Dataset(goes_files(cand[i])[3]) as nc:
            for b in BANDS:
                a = np.ma.filled(nc[f"CMI_C{b:02d}"][:], np.nan)
                acc[b].append(a[::4, ::4])          # subsample is plenty
    stats = {str(b): {"mean": float(np.nanmean(np.stack(acc[b]))),
                      "std": float(np.nanstd(np.stack(acc[b])))} for b in BANDS}
    STATS_PATH.write_text(json.dumps(stats, indent=2))
    print(f"computed and cached band stats -> {STATS_PATH}")

BAND_MEAN = np.array([stats[str(b)]["mean"] for b in BANDS], dtype=np.float32)
BAND_STD = np.array([stats[str(b)]["std"] for b in BANDS], dtype=np.float32)
for b, m, s in zip(BANDS, BAND_MEAN, BAND_STD):
    print(f"  band {b:>2}: mean {m:8.3f}  std {s:7.3f}")

## 4. Input part B — GLM lightning → a whole-day flash-density map

Lightning is a strong convective-storm signal. Flash floods are frequently
driven by **evening and overnight** convection that the daytime (16–21 UTC)
frames never see — so instead of a narrow per-frame window we summarize the
**entire day's** flashes into **one** map. We bin every flash of day D−1 onto
the same 2 km ABI pixel grid as the bands and `log1p`-scale the counts. This
requires projecting lat/lon → the GOES geostationary grid.

In [ ]:
import pyproj

# build the ABI fixed-grid geometry from the reference frame (GOES-16 in 2019)
with netCDF4.Dataset(files[0]) as nc:
    _p = nc["goes_imager_projection"]
    GEOS_WKT = pyproj.CRS.from_cf(
        {k: _p.getncattr(k) for k in _p.ncattrs()}).to_wkt()
    _h = float(_p.perspective_point_height)
    _xc = nc["x"][:].astype(np.float64) * _h        # column centres, metres (asc)
    _yc = nc["y"][:].astype(np.float64) * _h        # row centres, metres (desc)


def _edges(centers):
    d = np.diff(centers).mean()
    return np.concatenate([[centers[0] - d / 2], centers[:-1] + d / 2,
                           [centers[-1] + d / 2]])


X_EDGES = _edges(_xc)
Y_EDGES_ASC = _edges(_yc)[::-1].copy()              # ascending, for histogram2d
_TF = pyproj.Transformer.from_crs("EPSG:4326",
                                  pyproj.CRS.from_wkt(GEOS_WKT), always_xy=True)


def glm_day_map(flashes):
    "Whole-day flashes binned onto the ABI grid, log-scaled (one map per day)."
    if not len(flashes):
        return np.zeros((IMG_H, IMG_W), dtype=np.float32)
    X, Y = _TF.transform(flashes["lon"].to_numpy(np.float64),
                         flashes["lat"].to_numpy(np.float64))
    h2, _, _ = np.histogram2d(Y, X, bins=[Y_EDGES_ASC, X_EDGES])
    return (np.log1p(h2[::-1]) / 3.0).astype(np.float32)    # flip so row 0 = north


flashes = pd.read_parquet(glm_path(example_in_day), columns=["lat", "lon"])
glm = glm_day_map(flashes)
print(f"{len(flashes):,} flashes on {example_in_day} (whole day); "
      f"map shape {glm.shape}, {(glm > 0).sum():,} non-empty pixels")

plt.figure(figsize=(8, 4.6))
plt.imshow(glm[::4, ::4], cmap="magma")
plt.title(f"GLM whole-day flash density  {example_in_day}")
plt.colorbar(label="log1p(flash count)/3", shrink=0.8)
plt.xticks([]); plt.yticks([]); plt.tight_layout()

## 5. Assemble one full sample

Now we put it together. `load_sample(in_day, label_day)` reads the 6 frames,
normalizes the 6 bands, bins the whole day's lightning, and returns:

- `bands` of shape `(6, 6, 1500, 2500)` — the previous day's GOES sequence
- `glm` of shape `(1500, 2500)` — the previous day's whole-day flash map
- `y` of shape `(115, 186)` — that day's flood map

`np.nan_to_num` zeros out off-Earth/space pixels after normalization.

In [ ]:
def load_sample(in_day, label_day):
    files = goes_files(in_day)[:T_FRAMES]
    bands = np.zeros((T_FRAMES, N_BAND, IMG_H, IMG_W), dtype=np.float32)
    for t, f in enumerate(files):
        with netCDF4.Dataset(f) as nc:
            for c, b in enumerate(BANDS):
                a = np.ma.filled(nc[f"CMI_C{b:02d}"][:], np.nan)
                bands[t, c] = (a - BAND_MEAN[c]) / BAND_STD[c]   # normalize band
    np.nan_to_num(bands, copy=False)
    flashes = pd.read_parquet(glm_path(in_day), columns=["lat", "lon"])
    glm = glm_day_map(flashes)                                  # whole-day map
    return bands, glm, labels[label_day]


t0 = time.perf_counter()
bands, glm, y = load_sample(example_in_day, example_label_day)
dt = time.perf_counter() - t0
print(f"GOES seq shape {bands.shape}  dtype {bands.dtype}  = {bands.nbytes/1e6:.0f} MB")
print(f"GLM map  shape {glm.shape}  = {glm.nbytes/1e6:.1f} MB")
print(f"y        shape {y.shape}  = {int(y.sum())} flooded cells")
print(f"loaded one sample in {dt:.1f}s  (NetCDF decode + GLM binning)")

In [ ]:
# visualize: clean-IR across the 6 frames, the whole-day lightning, the label
fig, axes = plt.subplots(1, T_FRAMES + 2, figsize=(2.0 * (T_FRAMES + 2), 2.4))
for t in range(T_FRAMES):
    axes[t].imshow(bands[t, 4, ::6, ::6], cmap="gray_r")   # channel 4 = band 13
    axes[t].set_title(f"{_scan_dt(files[t]):%H:%M}Z", fontsize=8)
    axes[t].axis("off")
axes[T_FRAMES].imshow(glm[::6, ::6], cmap="magma")
axes[T_FRAMES].set_title("GLM whole-day", fontsize=8)
axes[T_FRAMES].axis("off")
axes[-1].imshow(np.where(land_mask, 0.15, np.nan), cmap="Greys", vmin=0, vmax=1)
axes[-1].imshow(np.where(land_mask, y, np.nan), cmap="Reds", vmin=0, vmax=1)
axes[-1].set_title(f"y: {example_label_day}", fontsize=8)
axes[-1].axis("off")
fig.suptitle("one sample:  band-13 sequence + whole-day lightning (D-1)   ->   "
             "floods (D)", fontsize=10)
plt.tight_layout()

## 6. Build the sample index

A valid sample needs **all 6 GOES frames** and a **GLM parquet** on the input
day (D−1), for every label day D in the year. Each sample is an independent
day-to-next-day prediction, so we split **randomly 70 / 20 / 10**
(train / val / test) with a fixed seed rather than holding out a time range.

> Caveat: adjacent days are weather-correlated, so a random split is a little
> more optimistic than a temporal one (a train day can sit right beside a test
> day). The seed keeps it reproducible; tweak `SPLIT_SEED` / `SPLIT_FRACS`.

In [ ]:
SPLIT_SEED = 0
SPLIT_FRACS = (0.70, 0.20, 0.10)            # train / val / test

rows = []
for d_label in sorted(labels):
    d_in = d_label - timedelta(days=1)
    if len(goes_files(d_in)) == T_FRAMES and glm_path(d_in).exists():
        rows.append({"in_day": pd.Timestamp(d_in),
                     "label_day": pd.Timestamp(d_label),
                     "n_pos": int(labels[d_label].sum())})

index = pd.DataFrame(rows)

# reproducible random 70/20/10 assignment
rng = np.random.default_rng(SPLIT_SEED)
perm = rng.permutation(len(index))
n_tr = int(SPLIT_FRACS[0] * len(index))
n_val = int(SPLIT_FRACS[1] * len(index))
split = np.array(["test"] * len(index), dtype=object)
split[perm[:n_tr]] = "train"
split[perm[n_tr:n_tr + n_val]] = "val"
index["split"] = split

print(index["split"].value_counts().reindex(["train", "val", "test"]).to_string())
print(f"\ntotal samples: {len(index)}")

# class balance over the train split -> pos_weight for notebook 02's loss
tr = index[index["split"] == "train"]
tr_y = np.stack([labels[d.date()] for d in tr["label_day"]])[:, land_mask]
pos_rate = float(tr_y.mean())
print(f"train positive rate: {pos_rate:.3%}  "
      f"-> suggested pos_weight ~= {min((1-pos_rate)/pos_rate, 100):.0f}")
index.head()

## 7. Materialize the prepared dataset (cache to disk)

Reading 6 NetCDFs + binning lightning costs a few seconds **per sample, per
epoch**. We pay that once: pre-compute every sample and write the GOES sequence
(`float16`, ~half the size), the whole-day GLM map, and the label as separate
`.npy` files. Notebook **02** then memory-maps these and trains ~10× faster.

This runs in parallel across many CPU cores. **Building all samples is heavy**
(~0.27 GB each), so it's gated behind `BUILD_CACHE`. By default we build a tiny
2-sample smoke test to prove the writer works and always write the manifest.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

CACHE_DIR.mkdir(parents=True, exist_ok=True)
index.to_parquet(CACHE_DIR / "manifest.parquet")        # splits for notebook 02
print(f"wrote manifest ({len(index)} samples) -> {CACHE_DIR}")
# (the grid + land mask are regenerated from config.build_grid_cells(), not stored)

BUILD_CACHE = True                  # <- flip to True to build ALL samples
N_WORKERS = 60                       # CPU processes (box has 64 cores)


def _write_one(args):
    in_day, label_day = args
    out = CACHE_DIR / f"{label_day:%Y%m%d}_x.npy"
    if out.exists():
        return 0
    bands, glm, y = load_sample(in_day, label_day)
    np.save(out, bands.astype(np.float16))                       # GOES sequence
    np.save(CACHE_DIR / f"{label_day:%Y%m%d}_glm.npy", glm.astype(np.float16))
    np.save(CACHE_DIR / f"{label_day:%Y%m%d}_y.npy", y.astype(np.uint8))
    return bands.nbytes // 2


todo = list(zip(index["in_day"].dt.date, index["label_day"].dt.date))
if not BUILD_CACHE:
    todo = todo[:2]
    print(f"smoke test: writing {len(todo)} samples "
          "(set BUILD_CACHE=True for all)")
else:
    gb = len(todo) * (bands.nbytes + glm.nbytes) / 2 / 1e9
    print(f"building ALL {len(todo)} samples (~{gb:.0f} GB float16) "
          f"on {N_WORKERS} workers ...")

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    written = list(ex.map(_write_one, todo))
print(f"done: {sum(b > 0 for b in written)} new files in "
      f"{(time.perf_counter()-t0)/60:.1f} min  -> {CACHE_DIR}")

---
### What we built

- **GOES** `(6, 6, 1500, 2500)` — 6 prev-day frames × 6 ABI bands
- **GLM** `(1500, 2500)` — one whole-day flash-density map
- **`y`** `(59, 95)` — next-day flood map on the 50 km grid, scored on the land mask
- a **manifest** (splits) + the materialized cache at
  `/mnt/disk1/datasets/floodnet_2019/` (`{date}_x.npy`, `{date}_glm.npy`, `{date}_y.npy`).
  The grid + land mask are **regenerated on the fly** from `config.build_grid_cells()`

**Next:** notebook **02** loads this cache, defines the U-Net + ConvLSTM, and
runs train / validate / test. Set `BUILD_CACHE = True` above and run the last
cell once before training on the full set.